# Import Libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans

from mpl_toolkits.mplot3d import Axes3D

plt.style.use("classic")
sns.set_palette("muted")

# Load Data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/krupalpatel07/dow-jones-global-market-shock-dataset/DJIA.csv")

df['Date'] = pd.to_datetime(df['Date'])

df = df.sort_values("Date")

df.set_index("Date", inplace=True)

# 1 Global Market Structure

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(df['Close'], linewidth=1.5)

plt.title("Dow Jones Global Market Structure (2000–Present)")

plt.xlabel("Year")

plt.ylabel("Index Level")

plt.show()

# 2 Interactive Market Explorer

In [ ]:
fig = go.Figure(data=[go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close']
)])

fig.update_layout(title="Dow Jones Interactive Market Structure")

fig.show()

# 3 Crisis Volatility Landscape

In [ ]:
returns = df['Close'].pct_change()

vol_matrix = []

window = 20

for i in range(len(returns)-window):
    vol_matrix.append(returns[i:i+window].values)

vol_matrix = np.array(vol_matrix)

plt.figure(figsize=(14,6))

sns.heatmap(vol_matrix.T,
            cmap="inferno")

plt.title("Market Volatility Landscape")

plt.show()

# 4 War Shock Detector

In [ ]:
returns = df['Close'].pct_change()

shock = abs(returns)

plt.figure(figsize=(15,6))

plt.plot(shock)

plt.title("Market Shock Detector")

plt.ylabel("Shock Intensity")

plt.show()

Large spikes = market panic.

# 5 Market Regime Detection

In [ ]:
features = pd.DataFrame()

features['returns'] = returns
features['volatility'] = returns.rolling(20).std()

features = features.dropna()

kmeans = KMeans(n_clusters=3)

features['regime'] = kmeans.fit_predict(features)

## Regime Map

In [ ]:
plt.figure(figsize=(15,6))

plt.scatter(features.index,
            df.loc[features.index]['Close'],
            c=features['regime'],
            cmap='plasma')

plt.title("Market Regime Detection")

plt.show()

# 6 Crash Drawdown Map

In [ ]:
cum = (1 + returns).cumprod()

peak = cum.cummax()

drawdown = (cum - peak) / peak

plt.figure(figsize=(15,5))

plt.fill_between(drawdown.index,
                 drawdown,
                 color="red",
                 alpha=0.6)

plt.title("Market Crash Landscape")

plt.show()

# 7 Momentum Surface (3D)

In [ ]:
fig = plt.figure(figsize=(10,7))

ax = fig.add_subplot(111, projection='3d')

x = np.arange(len(returns))
y = returns
z = returns.rolling(30).std()

ax.scatter(x,y,z,
           c=z,
           cmap="plasma")

ax.set_xlabel("Time")
ax.set_ylabel("Returns")
ax.set_zlabel("Volatility")

plt.title("Momentum Volatility Surface")

plt.show()

# 8 Interactive Risk Dashboard

In [ ]:
fig = px.line(df,
              x=df.index,
              y="Close",
              title="Dow Jones Interactive Market Dashboard")

fig.show()